# Inferência do roteador de sentimentos — EscutIA

Este notebook carrega o modelo base `Qwen/Qwen2.5-0.5B-Instruct` junto com o adapter LoRA do roteador produzido pelo notebook `Fine_Tuning_LoRA_EscutIA.ipynb`.

Antes de executar as células de inferência, o treinamento precisa ter terminado e o diretório `outputs/resultados/lora_escutia_router` precisa conter `adapter_config.json` e os pesos do adapter. A saída deste notebook é um JSON para ser encaminhado ao LLM principal; ela não é uma resposta conversacional final.

## 1. Preparar caminhos e dispositivo

A célula funciona tanto quando o notebook é aberto dentro de `EscutIA/fine_tuning_lora` quanto quando o Jupyter é iniciado na raiz do projeto.

In [3]:
from pathlib import Path
import json
import re
import torch
from IPython.display import display

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'configs').exists() and (BASE_DIR / 'EscutIA' / 'fine_tuning_lora' / 'configs').exists():
    BASE_DIR = BASE_DIR / 'EscutIA' / 'fine_tuning_lora'

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
ADAPTER_DIR = BASE_DIR / 'outputs' / 'resultados' / 'lora_escutia_router'

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(
        f'Adapter não encontrado em {ADAPTER_DIR}. Execute primeiro o fine-tuning e aguarde o término.'
    )
if not (ADAPTER_DIR / 'adapter_config.json').exists():
    raise FileNotFoundError(
        f'{ADAPTER_DIR} existe, mas não contém adapter_config.json. '
        'Verifique se o treinamento LoRA foi concluído e se o output_dir da configuração está correto.'
    )

if getattr(torch, 'cuda', None) is not None and torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif getattr(torch, 'xpu', None) is not None and torch.xpu.is_available():
    DEVICE = torch.device('xpu')
    DTYPE = torch.bfloat16
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    DTYPE = torch.float32
else:
    DEVICE = torch.device('cpu')
    DTYPE = torch.float32

print(f'Diretório base: {BASE_DIR}')
print(f'Adapter: {ADAPTER_DIR}')
print(f'Dispositivo: {DEVICE} | dtype: {DTYPE}')

Diretório base: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\fine_tuning_lora
Adapter: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\fine_tuning_lora\outputs\resultados\lora_escutia_router
Dispositivo: xpu | dtype: torch.bfloat16


## 2. Carregar o modelo base e o adapter LoRA

O adapter não é um modelo completo: ele precisa ser aplicado sobre o mesmo modelo base usado no treinamento.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

modelo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    torch_dtype=DTYPE,
)
modelo = PeftModel.from_pretrained(modelo_base, ADAPTER_DIR)
modelo = modelo.to(DEVICE)
modelo.eval()

print('Modelo base + adapter carregados com sucesso.')
print(f'Parâmetros treináveis restantes: {sum(p.numel() for p in modelo.parameters() if p.requires_grad):,}')

`torch_dtype` is deprecated! Use `dtype` instead!


Modelo base + adapter carregados com sucesso.
Parâmetros treináveis restantes: 0


## 3. Criar a função de roteamento

O formato da mensagem replica o formato conversacional preparado para o dataset: instrução de classificação, texto do usuário e geração de um JSON. A função valida o contrato antes de devolver o resultado ao LLM principal.

In [5]:
from contextlib import nullcontext

INSTRUCTION = 'Classifique o sentimento predominante do texto como negativo, neutro ou positivo e responda somente com um JSON válido no formato {"sentimento":"<rotulo>"}.'
SYSTEM_PROMPT = 'Você é um roteador de sentimentos. Responda somente com JSON válido no formato {"sentimento":"negativo|neutro|positivo"}.'
LABELS = ('negativo', 'neutro', 'positivo')

def _extrair_json(resposta: str):
    trecho = re.search(r'\{.*?\}', resposta, flags=re.DOTALL)
    if trecho is None:
        return None
    try:
        candidato = json.loads(trecho.group(0))
    except json.JSONDecodeError:
        return None
    if set(candidato) != {'sentimento'} or candidato['sentimento'] not in LABELS:
        return None
    return candidato

def classificar_sentimento(texto: str, max_new_tokens: int = 24, usar_adapter: bool = True):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError('Informe um texto não vazio.')

    mensagens = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'{INSTRUCTION}\n\nTexto: {texto.strip()}'},
    ]
    prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )
    entradas = tokenizer(prompt, return_tensors='pt')
    entradas = {nome: valor.to(DEVICE) for nome, valor in entradas.items()}

    contexto_adapter = nullcontext() if usar_adapter else modelo.disable_adapter()
    with contexto_adapter:
        with torch.inference_mode():
            saida = modelo.generate(
                **entradas,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

    novos_tokens = saida[0, entradas['input_ids'].shape[1]:]
    resposta = tokenizer.decode(novos_tokens, skip_special_tokens=True).strip()
    resposta_json = _extrair_json(resposta)
    return {
        'texto': texto,
        'resposta_bruta': resposta,
        'resposta_json': resposta_json,
        'sentimento': resposta_json['sentimento'] if resposta_json else None,
        'modelo_avaliado': 'adapter_lora' if usar_adapter else 'modelo_base',
    }

## 4. Testar exemplos

Execute esta célula para observar as respostas do adapter treinado. O campo `resposta_json` precisa ser preenchido; se for `None`, o roteador não respeitou o contrato e não deve encaminhar a resposta ao LLM principal.

In [6]:
exemplos = [
    'Estou muito feliz com a solução que encontrei para o problema.',
    'A reunião aconteceu conforme o planejado.',
    'Estou preocupado e frustrado com o resultado da prova.',
    'Hoje recebi uma notícia inesperada: preciso pensar melhor antes de responder.',
]

resultados = [classificar_sentimento(texto) for texto in exemplos]
display(__import__('pandas').DataFrame(resultados))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,texto,resposta_bruta,resposta_json,sentimento
0,Estou muito feliz com a solução que encontrei ...,"{""sentimento"":""positivo""}",{'sentimento': 'positivo'},positivo
1,A reunião aconteceu conforme o planejado.,"{""sentimento"":""positivo""}",{'sentimento': 'positivo'},positivo
2,Estou preocupado e frustrado com o resultado d...,"{""sentimento"":""negativo""}",{'sentimento': 'negativo'},negativo
3,Hoje recebi uma notícia inesperada: preciso pe...,"{""sentimento"":""negativo""}",{'sentimento': 'negativo'},negativo


## 5. Testar um texto próprio

Edite o texto abaixo e execute a célula. O campo `resposta_bruta` ajuda a identificar quando o modelo gerou algo diferente do JSON esperado. Encaminhe ao LLM principal somente o campo `resposta_json` validado.

In [9]:
texto_teste = 'Estou com raiva e não quero bater papo com ninguém hoje.'
resultado = classificar_sentimento(texto_teste)
print('Texto:', resultado['texto'])
print('Resposta do modelo:', resultado['resposta_bruta'])
print('JSON para o LLM principal:', json.dumps(resultado['resposta_json'], ensure_ascii=False) if resultado['resposta_json'] else 'inválido')
print('Sentimento identificado:', resultado['sentimento'] or 'não identificado')

Texto: Estou com raiva e não quero bater papo com ninguém hoje.
Resposta do modelo: {"sentimento":"negativo"}
JSON para o LLM principal: {"sentimento": "negativo"}
Sentimento identificado: negativo


## 6. Avaliar o roteador no conjunto congelado

Esta avaliação é opcional e pode executar centenas de gerações. Ela mede a taxa de JSON válido e a acurácia do campo `sentimento` usando `dados/preparados/escutia_evaluation.json`. Mantenha `EXECUTAR_AVALIACAO = False` até decidir executar essa etapa. Para comparar o modelo-base com o adapter, defina também `AVALIAR_BASELINE = True`; isso executa uma segunda rodada sem aplicar o adapter e registra as duas métricas no MLflow.

In [8]:
from collections import Counter
import hashlib
import mlflow
import pandas as pd

EXECUTAR_AVALIACAO = False
AVALIAR_BASELINE = False
CONFIRMACAO_AVALIACAO = ''
DATASET_DIR = (BASE_DIR.parent / 'dataset' / 'dados' / 'preparados').resolve()
EVALUATION_FILE = DATASET_DIR / 'escutia_evaluation.json'

if not EXECUTAR_AVALIACAO:
    print('Avaliação congelada não executada. Defina EXECUTAR_AVALIACAO = True para iniciar as gerações.')
else:
    if CONFIRMACAO_AVALIACAO != 'EXECUTAR_AVALIACAO_ROUTER':
        raise RuntimeError("Para executar, defina CONFIRMACAO_AVALIACAO = 'EXECUTAR_AVALIACAO_ROUTER'.")
    registros_avaliacao = json.loads(EVALUATION_FILE.read_text(encoding='utf-8'))
    resultados_avaliacao = []
    for registro in registros_avaliacao:
        esperado = json.loads(registro['output'])['sentimento']
        predicao = classificar_sentimento(registro['input'])
        resultados_avaliacao.append({
            'esperado': esperado,
            'predito': predicao['sentimento'],
            'json_valido': predicao['resposta_json'] is not None,
            'resposta_bruta': predicao['resposta_bruta'],
        })
    df_avaliacao = pd.DataFrame(resultados_avaliacao)
    acuracia = (df_avaliacao['esperado'] == df_avaliacao['predito']).mean()
    taxa_json = df_avaliacao['json_valido'].mean()
    display(df_avaliacao.head())
    display(pd.crosstab(df_avaliacao['esperado'], df_avaliacao['predito'], dropna=False))
    print(f'Acurácia do sentimento: {acuracia:.2%}')
    print(f'Taxa de JSON válido: {taxa_json:.2%}')

    acuracia_base = None
    taxa_json_base = None
    if AVALIAR_BASELINE:
        resultados_base = []
        for registro in registros_avaliacao:
            esperado = json.loads(registro['output'])['sentimento']
            predicao_base = classificar_sentimento(registro['input'], usar_adapter=False)
            resultados_base.append({
                'esperado': esperado,
                'predito': predicao_base['sentimento'],
                'json_valido': predicao_base['resposta_json'] is not None,
                'resposta_bruta': predicao_base['resposta_bruta'],
            })
        df_base = pd.DataFrame(resultados_base)
        acuracia_base = (df_base['esperado'] == df_base['predito']).mean()
        taxa_json_base = df_base['json_valido'].mean()
        print(f'Acurácia do modelo-base: {acuracia_base:.2%}')
        print(f'Taxa de JSON válido do modelo-base: {taxa_json_base:.2%}')

    avaliacao_dir = BASE_DIR / 'outputs' / 'avaliacao'
    avaliacao_dir.mkdir(parents=True, exist_ok=True)
    (avaliacao_dir / 'avaliacao_lora.json').write_text(
        json.dumps({
            'modelo': MODEL_NAME,
            'adapter': str(ADAPTER_DIR),
            'dataset': str(EVALUATION_FILE),
            'quantidade_amostras': len(df_avaliacao),
            'acuracia_sentimento': float(acuracia),
            'taxa_json_valido': float(taxa_json),
            'acuracia_modelo_base': float(acuracia_base) if acuracia_base is not None else None,
            'taxa_json_modelo_base': float(taxa_json_base) if taxa_json_base is not None else None,
        }, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    df_avaliacao.to_csv(avaliacao_dir / 'avaliacao_lora.csv', index=False, encoding='utf-8')

    dataset_hash = hashlib.sha256(EVALUATION_FILE.read_bytes()).hexdigest()
    mlflow.set_tracking_uri(f"sqlite:///{str(BASE_DIR.parent / 'mlflow.db').replace(chr(92), '/') }")
    mlflow.set_experiment('escutia-lora')
    with mlflow.start_run(run_name='avaliacao_lora_escutia') as run:
        metricas_mlflow = {
            'evaluation_accuracy': float(acuracia),
            'evaluation_valid_json_rate': float(taxa_json),
            'evaluation_samples': float(len(df_avaliacao)),
        }
        if acuracia_base is not None:
            metricas_mlflow['baseline_accuracy'] = float(acuracia_base)
            metricas_mlflow['baseline_valid_json_rate'] = float(taxa_json_base)
        mlflow.log_metrics(metricas_mlflow)
        mlflow.log_params({
            'evaluation_model': MODEL_NAME,
            'evaluation_adapter': str(ADAPTER_DIR),
            'evaluation_dataset_hash_sha256': dataset_hash,
        })
        mlflow.set_tags({
            'run_type': 'evaluation',
            'fine_tuning_type': 'lora',
            'evaluation_contract': 'sentimento_json',
        })
        mlflow.log_artifacts(str(avaliacao_dir), artifact_path='avaliacao')
        print(f'Avaliação registrada no MLflow: {run.info.run_id}')

Avaliação congelada não executada. Defina EXECUTAR_AVALIACAO = True para iniciar as gerações.
